<a href="https://colab.research.google.com/github/shira2718/my_portfolio/blob/master/mura_tmp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

from keras.preprocessing.image import load_img, img_to_array
from keras.utils import to_categorical
from keras.models import Sequential
from keras.layers import Dense, Dropout, Flatten
from keras.layers import Conv2D, MaxPooling2D

# ========= パラメータ設定 =========
# データディレクトリ（ラベルに応じたフォルダに画像が入っている前提）
basedir = "YOUR/DATA/PATH/**/*.jpg"  # 例: "./dataset/**/*.jpg"

# ラベルマッピング（フォルダ名が 0, 1, 2, ... となっている前提）
label_map = pd.Series(["0_Normal", "1_Mura-A", "2_Mura-B", "3_Mura-C", "4_Mura-D", "5_Circuit"])
num_labels = label_map.size

# 入力画像サイズ（任意調整）
hw = {"height": 266, "width": 218}

# ========= データ読み込み =========
featurelist = []
targetlist = []

files = glob.glob(basedir, recursive=True)
for imgfile in files:
    try:
        label = int(os.path.basename(os.path.dirname(imgfile)))
        img = load_img(imgfile, grayscale=False, target_size=(hw["height"], hw["width"]))
        array = img_to_array(img) / 255.0  # 正規化
        featurelist.append(array)
        targetlist.append(label)
    except Exception as e:
        print(f"スキップされたファイル: {imgfile}, 理由: {e}")

x_data = np.array(featurelist)
y_data = to_categorical(targetlist, num_labels)

# ========= データ分割 =========
x_train, x_test, y_train, y_test = train_test_split(
    x_data, y_data, stratify=y_data, train_size=0.8, random_state=1
)

# ========= CNNモデル構築 =========
model = Sequential()
model.add(Conv2D(32, (3, 3), activation='relu', input_shape=(hw["height"], hw["width"], 3)))
model.add(MaxPooling2D((2, 2)))
model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(MaxPooling2D((2, 2)))
model.add(Conv2D(128, (3, 3), activation='relu'))
model.add(MaxPooling2D((2, 2)))
model.add(Conv2D(128, (3, 3), activation='relu'))
model.add(MaxPooling2D((2, 2)))
model.add(Flatten())
model.add(Dropout(0.5))
model.add(Dense(512, activation='relu'))
model.add(Dense(num_labels, activation='softmax'))

model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

# ========= 学習 =========
epochs = 3
batch_size = 20

history = model.fit(
    x_train, y_train,
    epochs=epochs,
    batch_size=batch_size,
    validation_data=(x_test, y_test),
    verbose=1
)

# ========= モデル評価 =========
score = model.evaluate(x_test, y_test, verbose=0)
print("Test loss:", score[0])
print("Test accuracy:", score[1])


ValueError: With n_samples=0, test_size=None and train_size=0.8, the resulting train set will be empty. Adjust any of the aforementioned parameters.